In [2]:
# ============================================================
# Pyomo MILP (Cluster, Non-Sequential) + Monthly Cycle Closure
# CLEAN Single-Scenario (w0 / w1 / w2) for *UNCERTAIN* dataset
# - Reads scenario probability ω from Scenario_Prob (mandatory, with percent-fix)
# - Safe scenario prices (absolute or multiplier; sanity-checked)
# - Investment minima, P2P gating by RB build, anti buy/sell (z)
# - EV & Buildings can buy/sell to grid (toggleable)
# - Exports detailed objective breakdown: Invest / Operating / Degradation / TotalCost
# ============================================================

import os, numpy as np, pandas as pd
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, NonNegativeReals, Binary, Integers, Reals,
    Objective, Constraint, minimize, value, SolverFactory
)

# ------------------------ CONFIG ------------------------
DATA_XLSX_PATH = "p2p_ev_dataset_cluster_v2 _uncertain2.xlsx"  # ← توجه به فاصله قبل از "_uncertain"
RESULT_DIR     = "opt_results_singleW_uncertain"
P2P_FULL       = True
SELECTED_W     = "w01"    # سناریوی مورد نظر: w0 / w1 / w2

# حداقل‌های سرمایه‌گذاری
REQUIRED_STATIONS = 3
REQUIRED_RB       = 1

# آیا EVها مجازند مستقیم با شبکه تبادل کنند؟
ALLOW_EV_GRID = True

# پنالتی خیلی کوچک برای کاهش چرخه‌های P2P (اختیاری: 0 یعنی غیرفعال)
EPS_F = 0.0

# سناریوی قیمت‌ها: "auto" یا "off"
USE_SCENARIO_PRICES = "auto"
MAX_PRICE_FACTOR = 5.0            # سقف منطقی نسبت قیمت سناریویی به پایه
MAX_MULT_FACTOR_FOR_DG = 3.0      # سقف منطقی ضریب‌های Demand/Gen برای هشدار

# -------------------- Upload if needed --------------------
def must_file(path_hint):
    if os.path.exists(path_hint):
        return path_hint
    alt = path_hint.replace(" _uncertain", "_uncertain")  # اگر فایل بدون فاصله ذخیره شده باشد
    if alt != path_hint and os.path.exists(alt):
        return alt
    try:
        from google.colab import files
        print(f"⬆️ فایل `{path_hint}` را آپلود کنید ...")
        uploaded = files.upload()
        return list(uploaded.keys())[0]
    except Exception as e:
        raise FileNotFoundError(f"Dataset `{path_hint}` not found and no upload provided.") from e

DATA_XLSX_PATH = must_file(DATA_XLSX_PATH)
xls = pd.ExcelFile(DATA_XLSX_PATH)

# -------------------- Sets --------------------
B = pd.read_excel(xls,"Sets_Buildings")["Building"].astype(str).tolist()
V = pd.read_excel(xls,"Sets_EVs")["EV"].astype(str).tolist()
S = pd.read_excel(xls,"Sets_Stations")["Station"].astype(str).tolist()
R = pd.read_excel(xls,"Sets_RB")["RB"].astype(str).tolist()
M = pd.read_excel(xls,"Sets_Months")["Month"].tolist()
T = pd.read_excel(xls,"Sets_Timeslots")["Timeslot"].astype(str).tolist()

# -------------------- Base tables --------------------
Time_Delta = pd.read_excel(xls,"Time_Delta").set_index(["Timeslot","Month"])
Prices     = pd.read_excel(xls,"Prices").set_index(["Timeslot","Month"])

Demand_B = pd.read_excel(xls,"Demand_Buildings").set_index(["Building","Timeslot","Month"])
Gen_B    = pd.read_excel(xls,"Generation_Buildings").set_index(["Building","Timeslot","Month"])

EV_params = pd.read_excel(xls,"EV_Params").set_index("EV")
EV_Avh    = pd.read_excel(xls,"EV_Availability_Hours").set_index(["EV","Station","Timeslot","Month"])

RB_params = pd.read_excel(xls,"RB_Params").set_index("RB")
RB_invest = pd.read_excel(xls,"RB_Invest").set_index("RB")

Station_params = pd.read_excel(xls,"Station_Params").set_index("Station")
System_costs   = pd.read_excel(xls,"System_Costs").set_index("Param")

GridBuy = pd.read_excel(xls,"Grid_BuyCaps").set_index(["Node","Timeslot","Month"])
GridSell= pd.read_excel(xls,"Grid_SellCaps").set_index(["Node","Timeslot","Month"])
Fbar    = pd.read_excel(xls,"P2P_Fbar").set_index(["i","j"])

Theta_RB = pd.read_excel(xls,"ThroughputCaps_RB").set_index(["RB","Month"]) if "ThroughputCaps_RB" in xls.sheet_names else None
Theta_EV = pd.read_excel(xls,"ThroughputCaps_EV").set_index(["EV","Month"]) if "ThroughputCaps_EV" in xls.sheet_names else None

# -------------------- Scenario sheets (multipliers) --------------------
MulP  = pd.read_excel(xls,"ScenMul_Prices")   if "ScenMul_Prices"  in xls.sheet_names else None
MulDB = pd.read_excel(xls,"ScenMul_DemandB")  if "ScenMul_DemandB" in xls.sheet_names else None
MulGB = pd.read_excel(xls,"ScenMul_GenB")     if "ScenMul_GenB"    in xls.sheet_names else None
MulEV = pd.read_excel(xls,"ScenMul_EVHours")  if "ScenMul_EVHours" in xls.sheet_names else None

# ---------------- ENUMERATE SCENARIOS IN SHEETS (helper print) ----------------
def list_scenarios_in_sheet(xls, sheet, col="Scenario"):
    if sheet not in xls.sheet_names:
        return []
    df = pd.read_excel(xls, sheet)
    if col not in df.columns:
        return []
    return sorted(df[col].astype(str).str.strip().unique().tolist())

print("Scenarios found:")
print("  Scenario_Prob     :", list_scenarios_in_sheet(xls, "Scenario_Prob"))
print("  ScenMul_Prices    :", list_scenarios_in_sheet(xls, "ScenMul_Prices"))
print("  ScenMul_DemandB   :", list_scenarios_in_sheet(xls, "ScenMul_DemandB"))
print("  ScenMul_GenB      :", list_scenarios_in_sheet(xls, "ScenMul_GenB"))
print("  ScenMul_EVHours   :", list_scenarios_in_sheet(xls, "ScenMul_EVHours"))

# -------------------- EV Hours normalization (base) --------------------
EV_Avh_df = EV_Avh.reset_index()
Delta_df  = Time_Delta.reset_index()[["Timeslot","Month","Delta_hours"]]
EV_Avh_df = EV_Avh_df.merge(Delta_df, on=["Timeslot","Month"], how="left")

Hsum = EV_Avh_df.groupby(["EV","Timeslot","Month"], as_index=False)["Hours_Available"] \
                .sum().rename(columns={"Hours_Available":"Hsum"})
EV_Avh_df = EV_Avh_df.merge(Hsum, on=["EV","Timeslot","Month"], how="left")

eps = 1e-9
scale = np.where(EV_Avh_df["Hsum"]>0,
                 np.minimum(1.0, (EV_Avh_df["Delta_hours"]-eps)/EV_Avh_df["Hsum"]),
                 1.0)
EV_Avh_df["Hours_Available"] *= scale

# clamp نهایی
chk = EV_Avh_df.groupby(["EV","Timeslot","Month"], as_index=False)["Hours_Available"].sum() \
               .merge(Delta_df, on=["Timeslot","Month"], how="left")
viol = chk[chk["Hours_Available"] > chk["Delta_hours"] + 1e-6]
if len(viol):
    for _, row in viol.iterrows():
        mask = (EV_Avh_df["EV"]==row["EV"]) & (EV_Avh_df["Timeslot"]==row["Timeslot"]) & (EV_Avh_df["Month"]==row["Month"])
        factor = (row["Delta_hours"]-1e-9) / row["Hours_Available"]
        EV_Avh_df.loc[mask, "Hours_Available"] *= factor

EV_Avh_scaled = EV_Avh_df.set_index(["EV","Station","Timeslot","Month"])
Havail_base = EV_Avh_scaled["Hours_Available"].to_dict()

# -------------------- Base dicts --------------------
def dict_delta(): return {(t,m): float(Time_Delta.loc[(t,m),"Delta_hours"]) for t in T for m in M}
def dict_price_buy():  return {(t,m): float(Prices.loc[(t,m),"Pbuy"]) for t in T for m in M}
def dict_price_sell(): return {(t,m): float(Prices.loc[(t,m),"Psell"]) for t in T for m in M}

def dict_D_base():
    d={}
    for b in B:
        for t in T:
            for m in M:
                d[(b,t,m)] = float(Demand_B.loc[(b,t,m),"D_kWh"])
    for v in V:
        for t in T:
            for m in M:
                d[(v,t,m)] = 0.0
    for r in R:
        for t in T:
            for m in M:
                d[(r,t,m)] = 0.0
    return d

def dict_G_base():
    g={}
    for b in B:
        for t in T:
            for m in M:
                g[(b,t,m)] = float(Gen_B.loc[(b,t,m),"G_kWh"])
    for v in V:
        for t in T:
            for m in M:
                g[(v,t,m)] = 0.0
    for r in R:
        for t in T:
            for m in M:
                g[(r,t,m)] = 0.0
    return g

def dict_caps(df):
    return {idx: float(val) for idx,val in df.iloc[:,0].items()}

Pbuy_base, Psell_base, Delta_dict = dict_price_buy(), dict_price_sell(), dict_delta()
D_base, G_base = dict_D_base(), dict_G_base()
BuyCap_dict, SellCap_dict = dict_caps(GridBuy), dict_caps(GridSell)
Fcap_dict = {(i,j): float(Fbar.loc[(i,j),"Fcap_kWh"]) for (i,j) in Fbar.index}

# tech params
def evp(col):
    for c in [col, col.replace("KWh","kWh"), col.replace("kWh","KWh")]:
        if c in EV_params.columns: return EV_params[c].to_dict()
    raise KeyError(col)

eta_chV  = evp("eta_ch");       eta_disV = evp("eta_dis")
EminV    = evp("Emin_kWh");     EmaxV    = evp("Emax_kWh")
PchV     = evp("Pch_max_kW");   PdisV    = evp("Pdis_max_kW")
CdegV    = evp("Cdeg_per_kWh")

eta_chR  = RB_params["eta_ch"].to_dict()
eta_disR = RB_params["eta_dis"].to_dict()
EminR    = RB_params["Emin_kWh"].to_dict()
EmaxR    = RB_params["Emax_KWh" if "Emax_KWh" in RB_params.columns else "Emax_kWh"].to_dict()
PchR     = RB_params["Pch_max_kW"].to_dict()
PdisR    = RB_params["Pdis_max_kW"].to_dict()
CdegR    = RB_params["Cdeg_per_kWh"].to_dict()
C_battery= RB_invest["C_battery"].to_dict()

Kmax   = Station_params["Kmax"].to_dict()
Pport  = Station_params["Pport_kW"].to_dict()
Usite  = Station_params["Usite_kW"].to_dict()
C_station     = Station_params["C_station"].to_dict()
C_per_charger = Station_params["C_per_charger"].to_dict()
C_curt = float(System_costs.loc["C_curt","Value"])

ThetaR_dict = {(r,m): float(Theta_RB.loc[(r,m),"Theta_R_kWh"]) for (r,m) in (Theta_RB.index if Theta_RB is not None else [])}
ThetaV_dict = {(v,m): float(Theta_EV.loc[(v,m),"Theta_V_kWh"]) for (v,m) in (Theta_EV.index if Theta_EV is not None else [])}

# ---------------- READ ω WITH PERCENT FIX ----------------
def get_selected_prob_or_fail(xls, selected_w):
    if "Scenario_Prob" not in xls.sheet_names:
        raise ValueError("Sheet `Scenario_Prob` not found.")
    sp = pd.read_excel(xls, "Scenario_Prob")
    if "Scenario" not in sp.columns:
        raise ValueError("`Scenario_Prob` must include a `Scenario` column.")
    prob_col = "Prob" if "Prob" in sp.columns else sp.columns[-1]
    m = sp["Scenario"].astype(str).str.strip().str.lower() == str(selected_w).strip().lower()
    if not m.any():
        raise ValueError(f"Selected scenario `{selected_w}` not found in `Scenario_Prob`.")
    w = float(sp.loc[m, prob_col].iloc[0])
    if w > 1.0 and w <= 100.0:
        print(f"⚠ Scenario probability {w} looks like percent; interpreting as {w/100.0}.")
        w = w/100.0
    if not (0.0 <= w <= 1.0):
        raise ValueError(f"Scenario probability ω={w} out of [0,1]. Fix `Scenario_Prob`.")
    return w

OMEGA = get_selected_prob_or_fail(xls, SELECTED_W)
print(f"Selected scenario = {SELECTED_W}  |  ω = {OMEGA}")

# ---------------- SAFE SCENARIO PRICES ----------------
def price_for_selected_safe(t, m, which):
    base = Pbuy_base[(t,m)] if which=="buy" else Psell_base[(t,m)]
    if USE_SCENARIO_PRICES == "off" or MulP is None:
        return base

    df = MulP
    mask = (df.get("Scenario", pd.Series(dtype=str)).astype(str).str.strip().str.lower()
              == str(SELECTED_W).strip().lower())
    if "Timeslot" in df.columns:
        mask = mask & (df["Timeslot"].astype(str) == str(t))
    if "Month" in df.columns:
        mask = mask & (df["Month"].astype(str) == str(m))
    rows = df[mask]
    if rows.empty:
        return base

    col_abs = "Pbuy" if which=="buy" else "Psell"
    col_mul = "Mul_buy" if which=="buy" else ("Mul_sell" if "Mul_sell" in rows.columns else "Mul_buy")

    if col_abs in rows.columns and pd.notna(rows.iloc[0][col_abs]):
        val = float(rows.iloc[0][col_abs])
        if base > 0 and val/base > MAX_PRICE_FACTOR:
            print(f"⚠ {which} price too high at (t={t},m={m}): {val} vs base {base}. Using base.")
            return base
        return val

    if col_mul in rows.columns and pd.notna(rows.iloc[0][col_mul]):
        mul = float(rows.iloc[0][col_mul])
        if mul <= 0:
            print(f"⚠ {which} multiplier non-positive at (t={t},m={m}): {mul}. Using 1.0.")
            mul = 1.0
        if mul > MAX_PRICE_FACTOR:
            print(f"⚠ {which} multiplier too large at (t={t},m={m}): {mul}. Capping to {MAX_PRICE_FACTOR}.")
            mul = MAX_PRICE_FACTOR
        return base * mul

    return base

Pbuy_sel  = {(t,m): price_for_selected_safe(t,m,"buy")  for t in T for m in M}
Psell_sel = {(t,m): price_for_selected_safe(t,m,"sell") for t in T for m in M}

try:
    rb = [Pbuy_sel[(t,m)]/Pbuy_base[(t,m)]  if Pbuy_base[(t,m)]>0 else 1.0 for t in T for m in M]
    rs = [Psell_sel[(t,m)]/Psell_base[(t,m)] if Psell_base[(t,m)]>0 else 1.0 for t in T for m in M]
    print("Pbuy ratio min/median/max:", round(min(rb),3), "/", round(np.median(rb),3), "/", round(max(rb),3))
    print("Psell ratio min/median/max:", round(min(rs),3), "/", round(np.median(rs),3), "/", round(max(rs),3))
except Exception as _e:
    pass

# ---------------- SANITY for Demand/Generation multipliers ----------------
def mul_scalar(df_mul, key_cols):
    if df_mul is None:
        return 1.0
    df = df_mul
    mask = pd.Series([True]*len(df))
    for k,v in key_cols.items():
        if k not in df.columns:
            return 1.0
        mask = mask & (df[k].astype(str)==str(v))
    rows = df[mask]
    if (not rows.empty) and ("Mul" in rows.columns):
        return float(rows.iloc[0]["Mul"])
    return 1.0

def warn_if_big(mul, tag, key):
    if mul > MAX_MULT_FACTOR_FOR_DG:
        print(f"⚠ Large {tag} multiplier ({mul}) at {key}. Check `ScenMul_*` sheet.")

# ---------------- Scenario-adjusted demands, gens, EV hours ----------------
D_sel, G_sel = {}, {}
for b in B:
    for t in T:
        for m in M:
            md = mul_scalar(MulDB, {"Scenario":SELECTED_W,"Building":b,"Timeslot":t,"Month":m})
            mg = mul_scalar(MulGB, {"Scenario":SELECTED_W,"Building":b,"Timeslot":t,"Month":m})
            warn_if_big(md, "DemandB", (b,t,m))
            warn_if_big(mg, "GenB", (b,t,m))
            D_sel[(b,t,m)] = D_base[(b,t,m)] * md
            G_sel[(b,t,m)] = G_base[(b,t,m)] * mg

for v in V:
    for t in T:
        for m in M:
            D_sel[(v,t,m)] = 0.0; G_sel[(v,t,m)] = 0.0
for r in R:
    for t in T:
        for m in M:
            D_sel[(r,t,m)] = 0.0; G_sel[(r,t,m)] = 0.0

H_sel = {}
for v in V:
    for t in T:
        for m in M:
            base_list = [((v,s,t,m), float(Havail_base.get((v,s,t,m),0.0))) for s in S]
            scaled=[]
            for (key,val) in base_list:
                _, s, tt, mm = key
                mul = mul_scalar(MulEV, {"Scenario":SELECTED_W,"EV":v,"Station":s,"Timeslot":tt,"Month":mm})
                scaled.append((key, val*mul))
            Δ = Delta_dict[(t,m)]
            clipped = [ (key, max(0.0, min(vv, Δ))) for key,vv in scaled ]
            tot = sum(vv for _,vv in clipped)
            if tot > Δ + 1e-8:
                factor = (Δ - 1e-9)/tot
                clipped = [ (key, vv*factor) for key,vv in clipped ]
            for key,vv in clipped:
                v2,s2,_,_ = key
                H_sel[(v2,s2,t,m)] = vv

# -------------------- BUILD MODEL --------------------
def build_model():
    model = ConcreteModel()

    model.B = Set(initialize=B); model.V = Set(initialize=V)
    model.S = Set(initialize=S); model.R = Set(initialize=R)
    model.M = Set(initialize=M, ordered=True)
    model.T = Set(initialize=T, ordered=True)
    N = B + V + R
    model.N = Set(initialize=N)

    # index tuples
    model.ITM  = Set(initialize=[(i,t,m) for i in N for t in T for m in M], dimen=3)
    model.RTM  = Set(initialize=[(r,t,m) for r in R for t in T for m in M], dimen=3)
    model.VSTM = Set(initialize=[(v,s,t,m) for v in V for s in S for t in T for m in M], dimen=4)
    model.VTM  = Set(initialize=[(v,t,m) for v in V for t in T for m in M], dimen=3)
    if P2P_FULL:
        model.IJTM = Set(initialize=[(i,j,t,m) for i in N for j in N if i!=j for t in T for m in M], dimen=4)

    # Params (single scenario)
    model.Pbuy  = Param(model.T, model.M, initialize={(t,m): Pbuy_sel[(t,m)] for t in T for m in M},  within=Reals)
    model.Psell = Param(model.T, model.M, initialize={(t,m): Psell_sel[(t,m)] for t in T for m in M}, within=Reals)
    model.Delta = Param(model.T, model.M, initialize=Delta_dict, within=Reals)

    model.D = Param(model.ITM, initialize=D_sel, within=Reals)
    model.G = Param(model.ITM, initialize=G_sel, within=Reals)

    model.BuyCap  = Param(model.ITM, initialize=BuyCap_dict,  within=Reals)
    model.SellCap = Param(model.ITM, initialize=SellCap_dict, within=Reals)
    if P2P_FULL:
        model.Fcap = Param(model.N, model.N, initialize=lambda m,i,j: Fcap_dict.get((i,j),0.0), within=Reals)

    # Tech params
    model.eta_chV  = Param(model.V, initialize=eta_chV, within=Reals)
    model.eta_disV = Param(model.V, initialize=eta_disV, within=Reals)
    model.EminV    = Param(model.V, initialize=EminV, within=Reals)
    model.EmaxV    = Param(model.V, initialize=EmaxV, within=Reals)
    model.PchV     = Param(model.V, initialize=PchV, within=Reals)
    model.PdisV    = Param(model.V, initialize=PdisV, within=Reals)
    model.CdegV    = Param(model.V, initialize=CdegV, within=Reals)

    model.eta_chR  = Param(model.R, initialize=eta_chR, within=Reals)
    model.eta_disR = Param(model.R, initialize=eta_disR, within=Reals)
    model.EminR    = Param(model.R, initialize=EminR, within=Reals)
    model.EmaxR    = Param(model.R, initialize=EmaxR, within=Reals)
    model.PchR     = Param(model.R, initialize=PchR, within=Reals)
    model.PdisR    = Param(model.R, initialize=PdisR, within=Reals)
    model.CdegR    = Param(model.R, initialize=CdegR, within=Reals)

    # Investment & site
    model.C_station     = Param(model.S, initialize=C_station, within=Reals)
    model.C_per_charger = Param(model.S, initialize=C_per_charger, within=Reals)
    model.Kmax          = Param(model.S, initialize=Kmax, within=Reals)
    model.Pport         = Param(model.S, initialize=Pport, within=Reals)
    model.Usite         = Param(model.S, initialize=Usite, within=Reals)

    model.C_battery = Param(model.R, initialize=C_battery, within=Reals)
    model.C_curt    = Param(initialize=C_curt, within=Reals)

    # Availability (single scenario)
    def H_init(m, v, s, t, mn): return float(H_sel.get((v,s,t,mn), 0.0))
    model.HoursAvail = Param(model.VSTM, initialize=H_init, within=Reals)

    if len(ThetaR_dict):
        model.ThetaR = Param(model.R, model.M, initialize=lambda m,r,mn: ThetaR_dict[(r,mn)], within=Reals)
    if len(ThetaV_dict):
        model.ThetaV = Param(model.V, model.M, initialize=lambda m,v,mn: ThetaV_dict[(v,mn)], within=Reals)

    # -------- Decision vars --------
    model.x = Var(model.S, within=Binary)
    model.y = Var(model.R, within=Binary)
    model.k = Var(model.S, within=Integers, bounds=(0,None))

    model.Ebuy  = Var(model.ITM, within=NonNegativeReals)
    model.Esell = Var(model.ITM, within=NonNegativeReals)
    model.Lcurt = Var(model.ITM, within=NonNegativeReals)

    model.zBuySell = Var(model.ITM, within=Binary)  # anti-simultaneous

    if P2P_FULL:
        model.F = Var(model.IJTM, within=NonNegativeReals)

    model.CR = Var(model.RTM, within=NonNegativeReals)
    model.DR = Var(model.RTM, within=NonNegativeReals)

    model.CV = Var(model.VSTM, within=NonNegativeReals)
    model.DV = Var(model.VSTM, within=NonNegativeReals)

    # -------- Objective (use ω for stage-2) --------
    def obj_rule(m):
        invest = sum(m.C_station[s]*m.x[s] + m.C_per_charger[s]*m.k[s] for s in m.S) \
               + sum(m.C_battery[r]*m.y[r] for r in m.R)

        oper = sum(m.Pbuy[t,mn]*m.Ebuy[(i,t,mn)] - m.Psell[t,mn]*m.Esell[(i,t,mn)] + m.C_curt*m.Lcurt[(i,t,mn)]
                   for (i,t,mn) in m.ITM)
        degR = sum(m.CdegR[r]*(m.CR[(r,t,mn)] + m.DR[(r,t,mn)]) for (r,t,mn) in m.RTM)
        degV = sum(m.CdegV[v]*(sum(m.CV[(v,s,t,mn)] for s in m.S) + sum(m.DV[(v,s,t,mn)] for s in m.S))
                   for (v,t,mn) in m.VTM)

        penF = 0.0
        if P2P_FULL and EPS_F > 0:
            penF = EPS_F * sum(m.F[(i,j,t,mn)] for (i,j,t,mn) in m.IJTM)

        return invest + OMEGA * (oper + degR + degV + penF)
    model.TotalCost = Objective(rule=obj_rule, sense=minimize)

    # -------- Constraints --------
    # A) تعادل انرژی گره‌ها
    def balance_rule(m, i, t, mn):
        lhs = m.G[(i,t,mn)] + m.Ebuy[(i,t,mn)]
        rhs = m.D[(i,t,mn)] + m.Esell[(i,t,mn)] + m.Lcurt[(i,t,mn)]
        if P2P_FULL:
            lhs += sum(m.F[(j,i,t,mn)] for j in m.N if j!=i)
            rhs += sum(m.F[(i,j,t,mn)] for j in m.N if j!=i)
        if i in m.R:
            lhs += m.DR[(i,t,mn)]
            rhs += m.CR[(i,t,mn)]
        if i in m.V:
            lhs += sum(m.DV[(i,s,t,mn)] for s in m.S)
            rhs += sum(m.CV[(i,s,t,mn)] for s in m.S)
        return lhs == rhs
    model.Balance = Constraint(model.ITM, rule=balance_rule)

    # B) P2P ظرفیت + گِیت RB
    if P2P_FULL:
        def fcap_rule(m, i, j, t, mn):
            return m.F[(i,j,t,mn)] <= m.Fcap[i,j] if (i != j) else Constraint.Skip
        model.FcapCons = Constraint(model.IJTM, rule=fcap_rule)

        def F_gate_from_rule(m, i, j, t, mn):
            if i in m.R:
                return m.F[(i,j,t,mn)] <= m.Fcap[i,j] * m.y[i]
            return Constraint.Skip

        def F_gate_to_rule(m, i, j, t, mn):
            if j in m.R:
                return m.F[(i,j,t,mn)] <= m.Fcap[i,j] * m.y[j]
            return Constraint.Skip

        model.F_gate_from = Constraint(model.IJTM, rule=F_gate_from_rule)
        model.F_gate_to   = Constraint(model.IJTM, rule=F_gate_to_rule)

    # C) شبکه + گِیت RB + ضد خرید/فروش هم‌زمان
    def grid_buy_cap_rule(m, i, t, mn):
        if i in m.R:
            return m.Ebuy[(i, t, mn)] <= m.BuyCap[(i, t, mn)] * m.y[i]
        elif i in m.B:
            return m.Ebuy[(i, t, mn)] <= m.BuyCap[(i, t, mn)]
        elif i in m.V:
            return m.Ebuy[(i, t, mn)] <= (m.BuyCap[(i, t, mn)] if ALLOW_EV_GRID else 0.0)
        else:
            return m.Ebuy[(i, t, mn)] == 0.0

    def grid_sell_cap_rule(m, i, t, mn):
        if i in m.R:
            return m.Esell[(i, t, mn)] <= m.SellCap[(i, t, mn)] * m.y[i]
        elif i in m.B:
            return m.Esell[(i, t, mn)] <= m.SellCap[(i, t, mn)]
        elif i in m.V:
            return m.Esell[(i, t, mn)] <= (m.SellCap[(i, t, mn)] if ALLOW_EV_GRID else 0.0)
        else:
            return m.Esell[(i, t, mn)] == 0.0

    model.GridBuyCap  = Constraint(model.ITM, rule=grid_buy_cap_rule)
    model.GridSellCap = Constraint(model.ITM, rule=grid_sell_cap_rule)

    def no_simul_buy_rule(m, i, t, mn):
        return m.Ebuy[i,t,mn]  <= m.BuyCap[i,t,mn]  * m.zBuySell[i,t,mn]
    def no_simul_sell_rule(m, i, t, mn):
        return m.Esell[i,t,mn] <= m.SellCap[i,t,mn] * (1 - m.zBuySell[i,t,mn])

    model.NoSimulBuy  = Constraint(model.ITM, rule=no_simul_buy_rule)
    model.NoSimulSell = Constraint(model.ITM, rule=no_simul_sell_rule)

    # D) RB power->energy caps
    def rb_cap_ch_rule(m, r, t, mn):  return m.CR[(r,t,mn)] <= m.PchR[r]*m.Delta[t,mn]*m.y[r]
    def rb_cap_dis_rule(m, r, t, mn): return m.DR[(r,t,mn)] <= m.PdisR[r]*m.Delta[t,mn]*m.y[r]
    model.RB_CapCh  = Constraint(model.RTM, rule=rb_cap_ch_rule)
    model.RB_CapDis = Constraint(model.RTM, rule=rb_cap_dis_rule)

    # E) EV caps via Hours_Available
    def ev_cap_ch_rule(m, v, s, t, mn):  return m.CV[(v,s,t,mn)] <= m.PchV[v]*m.HoursAvail[(v,s,t,mn)]
    def ev_cap_dis_rule(m, v, s, t, mn): return m.DV[(v,s,t,mn)] <= m.PdisV[v]*m.HoursAvail[(v,s,t,mn)]
    model.EV_CapCh  = Constraint(model.VSTM, rule=ev_cap_ch_rule)
    model.EV_CapDis = Constraint(model.VSTM, rule=ev_cap_dis_rule)

    # F) Station links & capacities
    def station_k_vs_x_rule(m, s): return m.k[s] <= m.Kmax[s]*m.x[s]
    model.Station_k_vs_x = Constraint(model.S, rule=station_k_vs_x_rule)

    def station_portcap_rule(m, s, t, mn):
        return sum(m.CV[(v,s,t,mn)] + m.DV[(v,s,t,mn)] for v in m.V) <= m.k[s]*m.Pport[s]*m.Delta[t,mn]
    model.Station_PortCap = Constraint([(s,t,mn) for s in S for t in T for mn in M], rule=station_portcap_rule)

    def station_sitecap_rule(m, s, t, mn):
        return sum(m.CV[(v,s,t,mn)] + m.DV[(v,s,t,mn)] for v in m.V) <= m.Usite[s]*m.Delta[t,mn]*m.x[s]
    model.Station_SiteCap = Constraint([(s,t,mn) for s in S for t in T for mn in M], rule=station_sitecap_rule)

    # G) Monthly cycle closure
    def rb_cycle_rule(m, r, mn):
        return sum(m.eta_chR[r]*m.CR[(r,t,mn)] for t in m.T) == sum((1.0/m.eta_disR[r])*m.DR[(r,t,mn)] for t in m.T)
    model.RB_Cycle = Constraint(model.R, model.M, rule=rb_cycle_rule)

    def ev_cycle_rule(m, v, mn):
        lhs = sum(m.eta_chV[v]*sum(m.CV[(v,s,t,mn)] for s in m.S) for t in m.T)
        rhs = sum((1.0/m.eta_disV[v])*sum(m.DV[(v,s,t,mn)] for s in m.S) for t in m.T)
        return lhs == rhs
    model.EV_Cycle = Constraint(model.V, model.M, rule=ev_cycle_rule)

    # H) Throughput caps (optional)
    if len(ThetaR_dict):
        def rb_throughput_rule(m, r, mn):
            return sum(m.CR[(r,t,mn)] + m.DR[(r,t,mn)] for t in m.T) <= m.ThetaR[r,mn]
        model.RB_Throughput = Constraint(model.R, model.M, rule=rb_throughput_rule)

    if len(ThetaV_dict):
        def ev_throughput_rule(m, v, mn):
            return sum(sum(m.CV[(v,s,t,mn)] + m.DV[(v,s,t,mn)] for s in m.S) for t in m.T) <= m.ThetaV[v,mn]
        model.EV_Throughput = Constraint(model.V, model.M, rule=ev_throughput_rule)

    # I) Investment minima
    def min_stations_rule(m): return sum(m.x[s] for s in m.S) >= REQUIRED_STATIONS
    def min_rbs_rule(m):      return sum(m.y[r] for r in m.R) >= REQUIRED_RB
    def k_lower_rule(m, s):   return m.k[s] >= m.x[s]
    model.MinStations  = Constraint(rule=min_stations_rule)
    model.MinRBs       = Constraint(rule=min_rbs_rule)
    model.K_LowerBound = Constraint(model.S, rule=k_lower_rule)

    return model

model = build_model()

# -------------------- SOLVE --------------------
def get_solver():
    for cand in ["highs","cbc","glpk"]:
        try:
            if SolverFactory(cand).available(exception_flag=False):
                return SolverFactory(cand)
        except:
            pass
    raise RuntimeError("No MILP solver found. Install HiGHS/CBC/GLPK.")

solver = get_solver()
try:
    if 'highs' in str(solver):
        solver.options['mip_rel_gap'] = 1e-4
        solver.options['primal_feasibility_tolerance'] = 1e-9
        solver.options['dual_feasibility_tolerance']   = 1e-9
    if 'cbc' in str(solver):
        solver.options['ratioGap'] = 1e-4
        solver.options['integerTolerance'] = 1e-9
except:
    pass

res = solver.solve(model, tee=True)
print(f"[{SELECTED_W}] ->", res.solver.status, res.solver.termination_condition)
print("ω (probability) =", OMEGA)
print("OBJ (expected)  =", value(model.TotalCost))

# -------------------- Objective breakdown (compute & export) --------------------
def objective_breakdown(m, omega):
    invest = sum(value(m.C_station[s])*value(m.x[s]) +
                 value(m.C_per_charger[s])*value(m.k[s]) for s in m.S) \
           + sum(value(m.C_battery[r])*value(m.y[r]) for r in m.R)

    oper = sum(value(m.Pbuy[t,mn])*value(m.Ebuy[(i,t,mn)])
               - value(m.Psell[t,mn])*value(m.Esell[(i,t,mn)])
               + value(m.C_curt)*value(m.Lcurt[(i,t,mn)])
               for (i,t,mn) in m.ITM)

    degR = sum(value(m.CdegR[r])*(value(m.CR[(r,t,mn)]) + value(m.DR[(r,t,mn)]))
               for (r,t,mn) in m.RTM)

    degV = sum(value(m.CdegV[v])*(sum(value(m.CV[(v,s,t,mn)]) for s in m.S)
                                  + sum(value(m.DV[(v,s,t,mn)]) for s in m.S))
               for (v,t,mn) in m.VTM)

    degradation = degR + degV
    total = invest + omega*(oper + degradation)
    return invest, oper, degradation, total

invest, oper, deg, total = objective_breakdown(model, OMEGA)
print("==== Objective Breakdown ====")
print(f"Invest      : {invest:,.0f}")
print(f"Operating   : {oper:,.0f}")
print(f"Degradation : {deg:,.0f}")
print(f"TotalCost   : {total:,.0f}")

# -------------------- EXPORT --------------------
os.makedirs(RESULT_DIR, exist_ok=True)

def dump_var(var, headers, fname, tol=1e-12, kind="real"):
    rows=[]
    for key in var:
        v = value(var[key])
        if v is None:
            continue
        if kind=="binary":
            v = 1 if v >= 1-1e-6 else (0 if v <= 1e-6 else v)
        elif kind=="integer":
            v = int(round(v))
        if abs(v) < tol:
            v = 0.0
        rows.append(list(key)+[v] if isinstance(key, tuple) else [key, v])
    pd.DataFrame(rows, columns=headers+["value"]).to_csv(os.path.join(RESULT_DIR, fname), index=False)

suf = f"_{SELECTED_W}"

# Investment
dump_var(model.x, ["Station"], "x_station"+suf+".csv", kind="binary")
dump_var(model.k, ["Station"], "k_chargers"+suf+".csv", kind="integer")
dump_var(model.y, ["RB"],      "y_RB"+suf+".csv",      kind="binary")

# Grid & Curtailment
dump_var(model.Ebuy,  ["Node","Timeslot","Month"], "E_grid_buy"+suf+".csv")
dump_var(model.Esell, ["Node","Timeslot","Month"], "E_grid_sell"+suf+".csv")
dump_var(model.Lcurt, ["Node","Timeslot","Month"], "Load_curt"+suf+".csv")

# P2P
if P2P_FULL:
    dump_var(model.F, ["i","j","Timeslot","Month"], "F_p2p"+suf+".csv")

# RB
dump_var(model.CR, ["RB","Timeslot","Month"], "C_RB"+suf+".csv")
dump_var(model.DR, ["RB","Timeslot","Month"], "D_RB"+suf+".csv")

# EV
dump_var(model.CV, ["EV","Station","Timeslot","Month"], "C_EV"+suf+".csv")
dump_var(model.DV, ["EV","Station","Timeslot","Month"], "D_EV"+suf+".csv")

# Objective breakdown CSV
pd.DataFrame([{
    "Scenario": SELECTED_W,
    "Omega": OMEGA,
    "Invest": invest,
    "Operating": oper,
    "Degradation": deg,
    "TotalCost": total
}]).to_csv(os.path.join(RESULT_DIR, "objective_breakdown"+suf+".csv"), index=False)

print(f"✓ نتایج سناریو `{SELECTED_W}` با ω={OMEGA} در پوشه `{RESULT_DIR}` ذخیره شد.")


Scenarios found:
  Scenario_Prob     : ['W01', 'W02', 'W03']
  ScenMul_Prices    : ['W01', 'W02', 'W03']
  ScenMul_DemandB   : ['W01', 'W02', 'W03']
  ScenMul_GenB      : ['W01', 'W02', 'W03']
  ScenMul_EVHours   : ['W01', 'W02', 'W03']
Selected scenario = w01  |  ω = 0.5
Pbuy ratio min/median/max: 1.01 / 1.07 / 1.2
Psell ratio min/median/max: 1.01 / 1.07 / 1.2
Running HiGHS 1.11.0 (git hash: 364c83a): Copyright (c) 2025 HiGHS under MIT licence terms
MIP  has 298226 rows; 248863 cols; 897907 nonzeros; 3247 integer variables (3235 binary)
Coefficient ranges:
  Matrix [9e-01, 1e+06]
  Cost   [6e+00, 1e+09]
  Bound  [1e+00, 1e+00]
  RHS    [1e+00, 1e+09]
Presolving model
56846 rows, 237119 cols, 586163 nonzeros  0s
56846 rows, 152159 cols, 416243 nonzeros  2s
56846 rows, 152159 cols, 416243 nonzeros  3s

Solving MIP model with:
   56846 rows
   152159 cols (3235 binary, 12 integer, 0 implied int., 148912 continuous, 0 domain fixed)
   416243 nonzeros

Src: B => Branching; C => Central rou